# **Deskripsi Dataset**

Dataset yang digunakan merupakan gambaran tentang mahasiswa yang terdaftar dalam 
berbagai program sarjana yang ditawarkan oleh perguruan tinggi. Data ini mencakup data 
demografis, faktor sosial-ekonomi, dan informasi kinerja akademik yang dapat digunakan 
untuk menganalisis faktor-faktor yang mungkin memprediksi tingkat kesuksesan akademik 
mahasiswa.

# **Requirements and Config**

In [5]:
# !pip install seaborn matplotlib numpy pandas

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

class settings:
    DATA_DIR = "../dataset/"
    TRAIN_FILE = "train.csv"
    TEST_FILE = "test.csv"
    SUBMISSION_FILE = "sample_submission.csv"
    INDEX_COL = "Student_ID"
    SEED = 42
    TARGET = "Target"
    TRAIN_PATH = DATA_DIR + TRAIN_FILE
    TEST_PATH = DATA_DIR + TEST_FILE
    SUBMISSION_PATH = DATA_DIR + SUBMISSION_FILE

In [11]:
train = pd.read_csv(settings.TRAIN_PATH, index_col=settings.INDEX_COL)
test = pd.read_csv(settings.TEST_PATH, index_col=settings.INDEX_COL)
submission = pd.read_csv(settings.SUBMISSION_PATH, index_col=settings.INDEX_COL)
train.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Admission grade,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
Student_ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3743,1,17,1,9670,1,1,118.0,1,19,38,7,9,113.5,1,0,0,1,0,1,19,0,0,6,9,5,13.500000,0,0,6,6,6,14.000000,0,12.4,0.5,1.79,Graduate
3540,1,1,1,9070,1,1,139.0,1,1,19,3,9,134.9,1,0,0,1,1,0,20,0,0,6,7,6,13.666667,0,0,6,7,6,12.166667,1,16.2,0.3,-0.92,Graduate
1118,1,1,1,9500,1,1,138.0,1,38,19,9,5,144.3,0,0,0,1,0,0,20,0,0,7,9,6,12.700000,0,0,8,8,7,13.571429,0,13.9,-0.3,0.79,Graduate
791,1,17,1,9773,1,1,138.0,1,1,19,9,9,124.0,0,0,0,1,0,1,19,0,0,6,6,6,13.166667,0,0,6,6,6,13.833333,0,11.1,0.6,2.02,Graduate
4381,1,7,1,9500,1,2,140.0,1,38,37,7,8,140.0,0,0,0,1,0,0,29,0,0,8,14,4,11.325000,1,0,8,14,4,11.325000,1,12.7,3.7,-1.70,Dropout


# **Exploratory Data Analysis**

# **Preprocessing**

## **Data Cleaning**

In [ ]:
def drop_columns(df):
    col = []
    return df.drop(columns=col)

## **Data Transformation**

## **Feature Selection**

## **Dimensionality Reduction**

## **Pipeline**

In [ ]:
class Preprocessing:
    def fit(self, X):
        pass
    
    def transform(self, X):
        pass
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)
    
preprocessor = Preprocessing()

# **Modeling and Validation**

## **Metrics**

In [ ]:
from abc import ABC, abstractmethod

class Metric(ABC):
    """
    Metric interface: all metrics must implement __call__
    """

    @abstractmethod
    def __call__(self, y_true, y_pred):
        pass


# ============================================================
# Utility: Confusion Matrix
# ============================================================

class ConfusionMatrix:
    """
    Computes confusion matrix using NumPy.
    """

    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        labels = np.unique(np.concatenate([y_true, y_pred]))
        label_to_idx = {label: idx for idx, label in enumerate(labels)}
        
        cm = np.zeros((len(labels), len(labels)), dtype=int)

        for t, p in zip(y_true, y_pred):
            cm[label_to_idx[t], label_to_idx[p]] += 1
        
        return cm, labels


# ============================================================
# Accuracy
# ============================================================

class Accuracy(Metric):
    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        return float(np.mean(y_true == y_pred))


# ============================================================
# Precision Macro
# ============================================================

class PrecisionMacro(Metric):
    """
    Macro-averaged precision (one-vs-rest)
    """

    def __init__(self, include_pred_only=False):
        self.include_pred_only = include_pred_only

    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        if self.include_pred_only:
            labels = np.unique(np.concatenate([y_true, y_pred]))
        else:
            labels = np.unique(y_true)

        precisions = []

        for c in labels:
            tp = np.sum((y_true == c) & (y_pred == c))
            fp = np.sum((y_true != c) & (y_pred == c))

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            precisions.append(precision)

        return float(np.mean(precisions))


# ============================================================
# Recall Macro
# ============================================================

class RecallMacro(Metric):
    """
    Macro-averaged recall (one-vs-rest)
    """

    def __init__(self, include_pred_only=False):
        self.include_pred_only = include_pred_only

    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        if self.include_pred_only:
            labels = np.unique(np.concatenate([y_true, y_pred]))
        else:
            labels = np.unique(y_true)

        recalls = []

        for c in labels:
            tp = np.sum((y_true == c) & (y_pred == c))
            fn = np.sum((y_true == c) & (y_pred != c))

            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            recalls.append(recall)

        return float(np.mean(recalls))


# ============================================================
# F1 Macro
# ============================================================

class F1Macro(Metric):
    def __init__(self, include_pred_only=False):
        self.include_pred_only = include_pred_only

    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        if self.include_pred_only:
            labels = np.unique(np.concatenate([y_true, y_pred]))
        else:
            labels = np.unique(y_true)

        f1_scores = []

        for c in labels:
            tp = np.sum((y_true == c) & (y_pred == c))
            fp = np.sum((y_pred == c) & (y_true != c))
            fn = np.sum((y_true == c) & (y_pred != c))

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0

            if precision + recall == 0:
                f1 = 0.0
            else:
                f1 = 2 * precision * recall / (precision + recall)

            f1_scores.append(f1)

        return float(np.mean(f1_scores))


# ============================================================
# Metric Collection (Run multiple metrics at once)
# ============================================================

class MetricCollection:
    """
    Utility to compute multiple metrics at once.
    """

    def __init__(self, metrics: dict):
        """
        metrics: dict[name -> Metric instance]
        """
        self.metrics = metrics

    def __call__(self, y_true, y_pred):
        return {name: metric(y_true, y_pred) for name, metric in self.metrics.items()}


## **Validation Splitter**

In [ ]:
# ============================================================
# Base Class for K-Fold Variants
# ============================================================

class BaseKFold(ABC):

    def __init__(self, n_splits=5, shuffle=True, random_state=None):
        assert n_splits >= 2, "n_splits must be >= 2"
        self.n_splits = n_splits
        self.shuffle = shuffle
        self.random_state = random_state

    def _get_permutation(self, n_samples):
        if self.shuffle:
            rng = np.random.default_rng(self.random_state)
            return rng.permutation(n_samples)
        return np.arange(n_samples)

    @abstractmethod
    def split(self, X, y=None):
        pass


# ============================================================
# Standard K-Fold
# ============================================================

class KFold(BaseKFold):
    
    def split(self, X, y=None):
        X = np.asarray(X)
        n_samples = len(X)

        indices = self._get_permutation(n_samples)

        fold_sizes = np.full(self.n_splits, n_samples // self.n_splits, dtype=int)
        fold_sizes[: n_samples % self.n_splits] += 1

        current = 0
        for fold_size in fold_sizes:
            start, stop = current, current + fold_size
            test_index = indices[start:stop]
            train_index = np.concatenate((indices[:start], indices[stop:]))
            yield train_index, test_index
            current = stop


# ============================================================
# Stratified K-Fold
# ============================================================

class StratifiedKFold(BaseKFold):

    def split(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)

        indices = self._get_permutation(len(y))
        y_shuffled = y[indices]

        classes = np.unique(y_shuffled)
        class_indices = {cls: np.where(y_shuffled == cls)[0] for cls in classes}

        folds = [[] for _ in range(self.n_splits)]

        rng = np.random.default_rng(self.random_state)

        for cls in classes:
            cls_idx = class_indices[cls]

            if self.shuffle:
                cls_idx = rng.permutation(cls_idx)

            split_cls = np.array_split(cls_idx, self.n_splits)

            for i, batch in enumerate(split_cls):
                folds[i].extend(batch.tolist())

        for i in range(self.n_splits):
            test_local_idx = np.array(sorted(folds[i]))
            train_local_idx = np.array(sorted(list(set(range(len(y))) - set(test_local_idx))))

            yield indices[train_local_idx], indices[test_local_idx]


# ✅**Final Submission**